# GraphSAGE Hyperparameter Tuning

This notebook implements a distributed hyperparameter search for the GraphSAGE quantum error correction model.

**Aggregation:** Using `mean` and `max` only (LSTM removed due to 4x slower training time).

**Workflow:**
1. Set `WORKER_ID` below (1-5)
2. Run the training loop to train this worker's assigned configurations
3. After all workers complete, run the analysis cells

**Worker Assignment (50 configs total):**
- Worker 1: configs [0, 4, 8, 12, 16, 20, 24, 28, 32, 36]
- Worker 2: configs [1, 5, 9, 13, 17, 21, 25, 29, 33, 37]
- Worker 3: configs [2, 6, 10, 14, 18, 22, 26, 30, 34, 38]
- Worker 4: configs [3, 7, 11, 15, 19, 23, 27, 31, 35, 39]
- Worker 5: configs [40, 41, 42, 43, 44, 45, 46, 47, 48, 49] (temporary)

In [1]:
#==============================================================================
# WORKER CONFIGURATION - CHANGE THIS VALUE FOR EACH COLAB INSTANCE
#==============================================================================
WORKER_ID = 1  # Set to 1, 2, 3, 4, or 5 for each Colab instance
#==============================================================================

# Validate worker ID
assert WORKER_ID in [1, 2, 3, 4, 5], f"WORKER_ID must be 1, 2, 3, 4, or 5, got {WORKER_ID}"

# Show config assignment
if WORKER_ID == 5:
    # TEMPORARY: Worker 5 gets configs 40-49 directly
    my_config_ids = list(range(40, 50))
else:
    # Workers 1-4 use modulo assignment (configs 0-39)
    my_config_ids = [i for i in range(40) if i % 4 == (WORKER_ID - 1)]

print(f"Worker ID: {WORKER_ID}")
print(f"This worker will train {len(my_config_ids)} configs: {my_config_ids}")

Worker ID: 1
This worker will train 10 configs: [0, 4, 8, 12, 16, 20, 24, 28, 32, 36]


#### Imports

In [2]:
# Install required libraries (uncomment and run if needed)
!pip install stim pymatching numpy matplotlib torch tqdm networkx
!pip install torch_geometric

# For CUDA support with PyTorch Geometric, you may need:
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)


In [3]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from torch_geometric.loader import DataLoader

# Import from models.py
from models import (
    SurfaceCodeSampler,
    SparseGraph,
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

# Set up paths
TUNING_DIR = BASE_PATH / "gSAGE" / "tuning"
TUNING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = TUNING_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = TUNING_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = TUNING_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  TUNING_DIR: {TUNING_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  MODELS_DIR: {MODELS_DIR}")

KeyboardInterrupt: 

#### Generate Configs (ALREADY RUN)

Config generation has been completed. 50 configurations using `mean` and `max` aggregation only (LSTM removed).
The `configs.json` file already exists - this cell is for reference only.

In [ ]:
# # =============================================================================
# # CONFIG GENERATION - RUN ONCE ON ANY INSTANCE
# # =============================================================================

# CONFIGS_PATH = TUNING_DIR / "configs.json"

# # Check if configs already exist
# if CONFIGS_PATH.exists():
#     print(f"configs.json already exists at {CONFIGS_PATH}")
#     print("Skipping generation. Delete the file to regenerate.")
# else:
#     # Search space definition (LSTM removed - too slow)
#     SEARCH_SPACE = {
#         'num_layers': [2, 3, 4, 5],
#         'hidden_dim': [64, 128, 256, 512],
#         'learning_rate': [1e-4, 3e-4, 5e-4, 1e-3, 3e-3],
#         'dropout': [0.0, 0.1, 0.2, 0.3],
#         'aggr': ['mean', 'max']
#     }

#     # Fixed parameters (not searched)
#     FIXED_PARAMS = {
#         'batch_size': 256,
#         'epochs': 50,
#         'distance': 7,
#         'in_channels': 5
#     }

#     # Number of configurations to generate
#     N_CONFIGS = 50  # Updated from 40 to include Worker 5 configs
#     SEED = 42

#     # Set seed for reproducibility
#     random.seed(SEED)

#     # Generate random configurations
#     configs = []
#     for i in range(N_CONFIGS):
#         config = {
#             'id': i,
#             'num_layers': random.choice(SEARCH_SPACE['num_layers']),
#             'hidden_dim': random.choice(SEARCH_SPACE['hidden_dim']),
#             'learning_rate': random.choice(SEARCH_SPACE['learning_rate']),
#             'dropout': random.choice(SEARCH_SPACE['dropout']),
#             'aggr': random.choice(SEARCH_SPACE['aggr'])
#         }
#         configs.append(config)

#     # Create the full config document
#     config_doc = {
#         'seed': SEED,
#         'n_configs': N_CONFIGS,
#         'generated_at': datetime.now().isoformat(),
#         'search_space': SEARCH_SPACE,
#         'fixed_params': FIXED_PARAMS,
#         'configs': configs
#     }

#     # Save to file
#     with open(CONFIGS_PATH, 'w') as f:
#         json.dump(config_doc, f, indent=2)

#     print(f"Generated {N_CONFIGS} configurations with seed {SEED}")
#     print(f"Saved to: {CONFIGS_PATH}")
#     print(f"\nSearch space:")
#     for key, values in SEARCH_SPACE.items():
#         print(f"  {key}: {values}")
#     print(f"\nFixed parameters:")
#     for key, value in FIXED_PARAMS.items():
#         print(f"  {key}: {value}")
#     print(f"\nFirst 5 configs:")
#     for c in configs[:5]:
#         print(f"  {c}")

Generated 50 configurations (LSTM removed - using mean/max only)
Saved to: ../gSAGE/tuning/configs.json

Search space:
  num_layers: [2, 3, 4, 5]
  hidden_dim: [64, 128, 256, 512]
  learning_rate: [0.0001, 0.0003, 0.0005, 0.001, 0.003]
  dropout: [0.0, 0.1, 0.2, 0.3]
  aggr: ['mean', 'max']

Fixed parameters:
  batch_size: 256
  epochs: 50
  distance: 7
  in_channels: 5


#### Training Loop

This is the main training cell. It will:
1. Load the configs assigned to this worker
2. Skip any configs already completed (for resume support)
3. Train each config and save results incrementally

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_configs():
    """Load all configurations from configs.json."""
    configs_path = TUNING_DIR / "configs.json"
    if not configs_path.exists():
        raise FileNotFoundError(f"configs.json not found at {configs_path}. Run the config generation cell first.")
    with open(configs_path, 'r') as f:
        return json.load(f)

def get_worker_configs(all_configs, worker_id):
    """Get configs assigned to this worker using modulo assignment."""
    if worker_id == 5:
        # TEMPORARY: Worker 5 gets configs 40-49 directly
        return [c for c in all_configs['configs'] if 40 <= c['id'] <= 49]
    else:
        # Workers 1-4 use modulo assignment (configs 0-39)
        return [c for c in all_configs['configs'] if c['id'] < 40 and c['id'] % 4 == (worker_id - 1)]

def get_completed_ids(worker_id):
    """Get set of already-completed config IDs for this worker."""
    results_path = RESULTS_DIR / f"worker_{worker_id}.json"
    if not results_path.exists():
        return set()
    with open(results_path, 'r') as f:
        results = json.load(f)
    return {r['config_id'] for r in results if r.get('status') == 'completed'}

def save_result(result, worker_id):
    """Append a result to this worker's results file."""
    results_path = RESULTS_DIR / f"worker_{worker_id}.json"

    # Load existing results
    if results_path.exists():
        with open(results_path, 'r') as f:
            results = json.load(f)
    else:
        results = []

    # Append new result
    results.append(result)

    # Save back
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    return len(results)

def evaluate_model(model, graphs, device):
    """Evaluate model accuracy on a set of graphs."""
    model.model.eval()
    loader = DataLoader(graphs, batch_size=256, shuffle=False)

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model.model(batch)
            y = batch.y.float().view(-1, 1)
            correct += ((pred > 0.5).float() == y).sum().item()
            total += y.size(0)

    return correct / total if total > 0 else 0.0

def load_and_split_dataset(cache_name, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Load dataset from cache and split into train/val/test."""
    print(f"Loading dataset: {cache_name}")
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    cache.load(cache_name, verbose=True)
    graphs = cache.get_graphs()

    # Shuffle with fixed seed for reproducibility
    random.seed(seed)
    indices = list(range(len(graphs)))
    random.shuffle(indices)

    # Calculate split points
    n = len(graphs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    # Split
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    train_graphs = [graphs[i] for i in train_idx]
    val_graphs = [graphs[i] for i in val_idx]
    test_graphs = [graphs[i] for i in test_idx]

    print(f"Dataset split: {len(train_graphs)} train, {len(val_graphs)} val, {len(test_graphs)} test")

    return train_graphs, val_graphs, test_graphs

In [ ]:
# =============================================================================
# MAIN TRAINING LOOP
# =============================================================================

import gc

# Load all configurations
all_configs = load_configs()
fixed_params = all_configs['fixed_params']

# Get this worker's assigned configs
my_configs = get_worker_configs(all_configs, WORKER_ID)
print(f"\nWorker {WORKER_ID} assigned {len(my_configs)} configs:")
print(f"Config IDs: {[c['id'] for c in my_configs]}")

# Check which are already completed
completed_ids = get_completed_ids(WORKER_ID)
remaining_configs = [c for c in my_configs if c['id'] not in completed_ids]
print(f"\nAlready completed: {len(completed_ids)} configs")
print(f"Remaining: {len(remaining_configs)} configs")

if len(remaining_configs) == 0:
    print("\nAll configs for this worker are already completed!")
else:
    # Load dataset once (shared across all configs)
    cache_name = f"d{fixed_params['distance']}_baseline"
    train_graphs, val_graphs, test_graphs = load_and_split_dataset(cache_name)

In [ ]:
if len(remaining_configs) > 0:
    print(f"\n{'='*60}")
    print(f"Starting training for Worker {WORKER_ID}")
    print(f"{'='*60}")

    for i, config in enumerate(remaining_configs):
        config_id = config['id']
        print(f"\n{'='*60}")
        print(f"Config {config_id} ({i+1}/{len(remaining_configs)})")
        print(f"{'='*60}")
        print(f"Parameters:")
        print(f"  num_layers: {config['num_layers']}")
        print(f"  hidden_dim: {config['hidden_dim']}")
        print(f"  learning_rate: {config['learning_rate']}")
        print(f"  dropout: {config['dropout']}")
        print(f"  aggr: {config['aggr']}")

        start_time = time.time()

        try:
            # Initialize model with config hyperparameters
            model = GraphSAGE(
                nickname=f"config_{config_id}",
                in_channels=fixed_params['in_channels'],
                hidden_dim=config['hidden_dim'],
                num_layers=config['num_layers'],
                dropout=config['dropout'],
                aggr=config['aggr'],
                device=device,
                base_path=BASE_PATH,
                seed=config_id  # Use config_id as seed for reproducibility
            )

            # Train the model
            epoch_losses = model.train(
                graphs=train_graphs,
                epochs=fixed_params['epochs'],
                batch_size=fixed_params['batch_size'],
                lr=config['learning_rate'],
                verbose=True
            )

            # Evaluate on validation and test sets
            val_accuracy = evaluate_model(model, val_graphs, device)
            test_accuracy = evaluate_model(model, test_graphs, device)

            training_time = time.time() - start_time

            # Save the model to tuning/models/
            model_path = MODELS_DIR / f"config_{config_id}.pt"
            save_dict = {
                'state_dict': model.model.state_dict(),
                'config': model._config,
                'hyperparams': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'timestamp': datetime.now().isoformat()
            }
            torch.save(save_dict, model_path)
            print(f"\nModel saved to: {model_path}")

            # Create result record
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': val_accuracy,
                'test_accuracy': test_accuracy,
                'training_time_sec': training_time,
                'final_loss': epoch_losses[-1] if epoch_losses else None,
                'model_path': str(model_path),
                'status': 'completed',
                'timestamp': datetime.now().isoformat()
            }

            print(f"\nResults:")
            print(f"  Val Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
            print(f"  Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
            print(f"  Training Time: {training_time:.1f}s ({training_time/60:.1f} min)")

        except Exception as e:
            training_time = time.time() - start_time
            print(f"\nERROR: Config {config_id} failed: {str(e)}")
            import traceback
            traceback.print_exc()
            result = {
                'config_id': config_id,
                'config': config,
                'val_accuracy': None,
                'test_accuracy': None,
                'training_time_sec': training_time,
                'final_loss': None,
                'model_path': None,
                'status': 'failed',
                'error': str(e),
                'timestamp': datetime.now().isoformat()
            }
            # Cleanup on error too
            if 'model' in dir():
                del model

        # Save result incrementally
        n_saved = save_result(result, WORKER_ID)
        print(f"Result saved ({n_saved} total for this worker)")

        # Aggressive cleanup to prevent RAM/VRAM from filling up
        if 'model' in dir():
            del model
        if 'epoch_losses' in dir():
            del epoch_losses
        if 'save_dict' in dir():
            del save_dict

        # Force garbage collection
        gc.collect()

        # Clear CUDA cache if available
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(f"Memory cleared.")

    print(f"\n{'='*60}")
    print(f"Worker {WORKER_ID} COMPLETE!")
    print(f"{'='*60}")
    print(f"Total configs trained: {len(remaining_configs)}")
    print(f"Results saved to: {RESULTS_DIR / f'worker_{WORKER_ID}.json'}")
    print(f"Models saved to: {MODELS_DIR}")

#### Combine Results (RUN AFTER ALL WORKERS COMPLETE)

This cell combines results from all 5 workers into a single CSV file and identifies the best configurations.

In [ ]:
# # =============================================================================
# # COMBINE RESULTS FROM ALL WORKERS
# # =============================================================================

# def combine_all_results():
#     """Load and combine results from all 5 workers."""
#     all_results = []

#     for worker_id in [1, 2, 3, 4, 5]:
#         results_path = RESULTS_DIR / f"worker_{worker_id}.json"
#         if results_path.exists():
#             with open(results_path, 'r') as f:
#                 worker_results = json.load(f)
#             all_results.extend(worker_results)
#             print(f"Worker {worker_id}: {len(worker_results)} results")
#         else:
#             print(f"Worker {worker_id}: No results file found")

#     return all_results

# # Combine results
# print("Combining results from all workers...")
# all_results = combine_all_results()

# # Filter to completed only
# completed_results = [r for r in all_results if r.get('status') == 'completed']
# failed_results = [r for r in all_results if r.get('status') == 'failed']

# print(f"\nTotal results: {len(all_results)}")
# print(f"  Completed: {len(completed_results)}")
# print(f"  Failed: {len(failed_results)}")

# if len(completed_results) > 0:
#     # Create DataFrame
#     df = pd.DataFrame(completed_results)

#     # Flatten config dict into columns
#     config_df = pd.json_normalize(df['config'])
#     config_df.columns = [f"hp_{col}" for col in config_df.columns]
#     df = pd.concat([df.drop('config', axis=1), config_df], axis=1)

#     # Sort by validation accuracy
#     df = df.sort_values('val_accuracy', ascending=False).reset_index(drop=True)

#     # Save to CSV
#     csv_path = TUNING_DIR / "combined_results.csv"
#     df.to_csv(csv_path, index=False)
#     print(f"\nResults saved to: {csv_path}")

#     # Print top 5 configurations
#     print(f"\n{'='*60}")
#     print("TOP 5 CONFIGURATIONS (by validation accuracy)")
#     print(f"{'='*60}")

#     for i, row in df.head(5).iterrows():
#         print(f"\n#{i+1} - Config {row['config_id']}")
#         print(f"  Val Accuracy:  {row['val_accuracy']:.4f} ({row['val_accuracy']*100:.2f}%)")
#         print(f"  Test Accuracy: {row['test_accuracy']:.4f} ({row['test_accuracy']*100:.2f}%)")
#         print(f"  Training Time: {row['training_time_sec']:.1f}s")
#         print(f"  Hyperparameters:")
#         print(f"    num_layers: {row['hp_num_layers']}")
#         print(f"    hidden_dim: {row['hp_hidden_dim']}")
#         print(f"    learning_rate: {row['hp_learning_rate']}")
#         print(f"    dropout: {row['hp_dropout']}")
#         print(f"    aggr: {row['hp_aggr']}")

#     # Best configuration summary
#     best = df.iloc[0]
#     print(f"\n{'='*60}")
#     print("BEST CONFIGURATION")
#     print(f"{'='*60}")
#     print(f"Config ID: {best['config_id']}")
#     print(f"Val Accuracy: {best['val_accuracy']:.4f}")
#     print(f"Test Accuracy: {best['test_accuracy']:.4f}")
#     print(f"\nHyperparameters for production use:")
#     print(f"  num_layers = {best['hp_num_layers']}")
#     print(f"  hidden_dim = {best['hp_hidden_dim']}")
#     print(f"  learning_rate = {best['hp_learning_rate']}")
#     print(f"  dropout = {best['hp_dropout']}")
#     print(f"  aggr = '{best['hp_aggr']}'")

#     # Overall statistics
#     print(f"\n{'='*60}")
#     print("OVERALL STATISTICS")
#     print(f"{'='*60}")
#     print(f"Val Accuracy:  mean={df['val_accuracy'].mean():.4f}, std={df['val_accuracy'].std():.4f}")
#     print(f"Test Accuracy: mean={df['test_accuracy'].mean():.4f}, std={df['test_accuracy'].std():.4f}")
#     print(f"Training Time: mean={df['training_time_sec'].mean():.1f}s, total={df['training_time_sec'].sum()/3600:.1f}h")
# else:
#     print("\nNo completed results to analyze.")

#### Analysis & Visualization

Generate plots to understand hyperparameter importance and find patterns.

In [ ]:
# # =============================================================================
# # ANALYSIS & VISUALIZATION
# # =============================================================================

# # Load results if not already loaded
# if 'df' not in dir() or df is None:
#     csv_path = TUNING_DIR / "combined_results.csv"
#     if csv_path.exists():
#         df = pd.read_csv(csv_path)
#         print(f"Loaded {len(df)} results from {csv_path}")
#     else:
#         raise FileNotFoundError("Run the combine results cell first!")

# # Create figure with subplots
# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
# fig.suptitle('GraphSAGE Hyperparameter Tuning Analysis', fontsize=14, fontweight='bold')

# # 1. Box plot: Validation accuracy by num_layers
# ax = axes[0, 0]
# df.boxplot(column='val_accuracy', by='hp_num_layers', ax=ax)
# ax.set_title('Accuracy by Number of Layers')
# ax.set_xlabel('Number of Layers')
# ax.set_ylabel('Validation Accuracy')
# plt.sca(ax)
# plt.xticks(rotation=0)

# # 2. Box plot: Validation accuracy by hidden_dim
# ax = axes[0, 1]
# df.boxplot(column='val_accuracy', by='hp_hidden_dim', ax=ax)
# ax.set_title('Accuracy by Hidden Dimension')
# ax.set_xlabel('Hidden Dimension')
# ax.set_ylabel('Validation Accuracy')

# # 3. Box plot: Validation accuracy by learning_rate
# ax = axes[0, 2]
# # Format learning rates for display
# df['lr_label'] = df['hp_learning_rate'].apply(lambda x: f'{x:.0e}')
# df.boxplot(column='val_accuracy', by='lr_label', ax=ax)
# ax.set_title('Accuracy by Learning Rate')
# ax.set_xlabel('Learning Rate')
# ax.set_ylabel('Validation Accuracy')

# # 4. Box plot: Validation accuracy by dropout
# ax = axes[1, 0]
# df.boxplot(column='val_accuracy', by='hp_dropout', ax=ax)
# ax.set_title('Accuracy by Dropout')
# ax.set_xlabel('Dropout')
# ax.set_ylabel('Validation Accuracy')

# # 5. Box plot: Validation accuracy by aggregation
# ax = axes[1, 1]
# df.boxplot(column='val_accuracy', by='hp_aggr', ax=ax)
# ax.set_title('Accuracy by Aggregation Function')
# ax.set_xlabel('Aggregation')
# ax.set_ylabel('Validation Accuracy')

# # 6. Speed vs Accuracy tradeoff
# ax = axes[1, 2]
# scatter = ax.scatter(df['training_time_sec']/60, df['val_accuracy'],
#                      c=df['hp_hidden_dim'], cmap='viridis', alpha=0.7, s=50)
# ax.set_xlabel('Training Time (minutes)')
# ax.set_ylabel('Validation Accuracy')
# ax.set_title('Speed vs Accuracy Tradeoff')
# plt.colorbar(scatter, ax=ax, label='Hidden Dim')

# # Mark top 5
# top5 = df.head(5)
# ax.scatter(top5['training_time_sec']/60, top5['val_accuracy'],
#            c='red', s=100, marker='*', label='Top 5', zorder=5)
# ax.legend()

# plt.tight_layout()
# plt.savefig(PLOTS_DIR / 'hyperparameter_analysis.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"\nPlot saved to: {PLOTS_DIR / 'hyperparameter_analysis.png'}")

In [ ]:
# # =============================================================================
# # HYPERPARAMETER IMPORTANCE ANALYSIS
# # =============================================================================

# print("Hyperparameter Importance Analysis")
# print("="*60)
# print("\nMean validation accuracy by hyperparameter value:\n")

# # Analyze each hyperparameter
# hyperparams = ['hp_num_layers', 'hp_hidden_dim', 'hp_learning_rate', 'hp_dropout', 'hp_aggr']

# importance_scores = {}

# for hp in hyperparams:
#     print(f"\n{hp.replace('hp_', '').upper()}:")
#     grouped = df.groupby(hp)['val_accuracy'].agg(['mean', 'std', 'count'])
#     grouped = grouped.sort_values('mean', ascending=False)

#     # Calculate importance as range of means
#     importance = grouped['mean'].max() - grouped['mean'].min()
#     importance_scores[hp] = importance

#     for idx, row in grouped.iterrows():
#         print(f"  {idx}: mean={row['mean']:.4f}, std={row['std']:.4f}, n={int(row['count'])}")
#     print(f"  Range: {importance:.4f}")

# # Rank hyperparameters by importance
# print(f"\n{'='*60}")
# print("HYPERPARAMETER IMPORTANCE RANKING")
# print("(by range of mean accuracy across values)")
# print(f"{'='*60}")

# sorted_importance = sorted(importance_scores.items(), key=lambda x: x[1], reverse=True)
# for i, (hp, score) in enumerate(sorted_importance, 1):
#     hp_name = hp.replace('hp_', '')
#     print(f"{i}. {hp_name}: {score:.4f}")

# # Plot importance
# fig, ax = plt.subplots(figsize=(10, 5))
# names = [hp.replace('hp_', '') for hp, _ in sorted_importance]
# scores = [score for _, score in sorted_importance]
# bars = ax.barh(names, scores, color='steelblue')
# ax.set_xlabel('Importance (Range of Mean Accuracy)')
# ax.set_title('Hyperparameter Importance')
# ax.invert_yaxis()

# # Add value labels
# for bar, score in zip(bars, scores):
#     ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
#             f'{score:.4f}', va='center', fontsize=10)

# plt.tight_layout()
# plt.savefig(PLOTS_DIR / 'hyperparameter_importance.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"\nPlot saved to: {PLOTS_DIR / 'hyperparameter_importance.png'}")

#### Load Best Model

Use this cell to load a specific trained model for further use.

In [ ]:
# # =============================================================================
# # LOAD A SPECIFIC MODEL
# # =============================================================================

# def load_tuning_model(config_id):
#     """Load a trained model from the tuning results."""
#     model_path = MODELS_DIR / f"config_{config_id}.pt"

#     if not model_path.exists():
#         raise FileNotFoundError(f"Model not found: {model_path}")

#     # Load saved data
#     save_dict = torch.load(model_path, map_location=device, weights_only=False)

#     # Get hyperparameters
#     hp = save_dict['hyperparams']
#     config = save_dict['config']

#     print(f"Loading model from config {config_id}")
#     print(f"  Val Accuracy: {save_dict['val_accuracy']:.4f}")
#     print(f"  Test Accuracy: {save_dict['test_accuracy']:.4f}")
#     print(f"  Hyperparameters: layers={hp['num_layers']}, dim={hp['hidden_dim']}, "
#           f"lr={hp['learning_rate']}, dropout={hp['dropout']}, aggr={hp['aggr']}")

#     # Create model with correct architecture
#     model = GraphSAGE(
#         nickname=f"tuning_config_{config_id}",
#         in_channels=config['in_channels'],
#         hidden_dim=config['hidden_dim'],
#         num_layers=config['num_layers'],
#         dropout=config.get('dropout', 0.0),
#         aggr=config.get('aggr', 'mean'),
#         device=device,
#         base_path=BASE_PATH
#     )

#     # Load weights
#     model.model.load_state_dict(save_dict['state_dict'])
#     model.model.eval()

#     return model, save_dict

# # Example: Load the best model (config ID from top of rankings)
# # Uncomment and set the config_id to load:

# # best_config_id = 0  # Replace with actual best config ID from analysis
# # best_model, best_info = load_tuning_model(best_config_id)

# # List available models
# print("Available models:")
# for model_file in sorted(MODELS_DIR.glob("config_*.pt")):
#     print(f"  {model_file.name}")